# DTAT351. deeptrack.utils

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/DTAT351_utils.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the module [deeptrack.utils](../../deeptrack/utils.py).

In [2]:
from deeptrack import utils

## 1. What is `utils.py`?

The `utils.py` module is DeepTrack2’s toolbox for generic utility functions that enhance code clarity, robustness, and developer productivity.

It offers a set of reusable building blocks for introspection, type safety, and argument management, making it easier to write flexible, backend-agnostic code and advanced scientific pipelines.

The key roles of `utils.py` are:

- **Method Detection (`hasmethod()`):**
    It provides a universal way to check if any Python object has a specific callable method (such as `.predict()`, `.forward()`, or other interface elements), supporting polymorphism and dynamic APIs.

- **Safe List Conversion (`as_list()`):**
    It ensures that any object—scalar, iterable, array, or tensor—can be handled uniformly as a list. This is especially useful for data preprocessing and batch handling.

- **Function Signature Introspection (`get_kwarg_names()`):**
    It enables inspection of which arguments a function accepts, including detection of keyword-only and default arguments. This is essential for metaprogramming, adapters, and dynamic pipelines.

- **Default Argument Checking (`kwarg_has_default()`):**
    It allows you to programmatically determine whether a function parameter has a default value, improving error handling and documentation generation.

- **Safe Function Calling (`safe_call()`):**
    It facilitates calling arbitrary functions with dictionaries of arguments, ensuring that only valid and accepted arguments are passed—making it easy to build wrappers, hooks, and dynamic dispatchers.

## 2.  Dynamically Detecting Methods with `has_method()`

The `hasmethod(obj, method_name)` function checks whether a given object has an attribute with the specified name and that the attribute is callable.

In [3]:
from deeptrack.utils import hasmethod

It works with a user-defined class ...

In [4]:
# User-defined class
class Model:
    def predict(self, x):
        return x

In [5]:
model = Model()
hasmethod(model, "predict")

True

In [6]:
hasmethod(model, "fit")

False

... with built-in types ...

In [7]:
hasmethod([1, 2, 3], "append")

True

In [8]:
hasmethod([1, 2, 3], "nonexistent")

False

... and with modules.

In [9]:
import math

print(hasmethod(math, "sqrt"))

True


In [10]:
hasmethod(math, "fake")

False

## 3. Converting to Lists with `as_list()`

The `as_list(obj)` function converts any object into a list, handling a wide range of types and edge cases.

In [11]:
from deeptrack.utils import as_list

Scalars and non-iterables are wrapped in a list.

In [12]:
as_list(5)

[5]

In [13]:
as_list(None)

[None]

Iterables (lists, tuples, sets, generators, numpy arrays, PyTorch tensors) are converted to lists.

In [14]:
as_list([1, 2, 3])

[1, 2, 3]

In [15]:
as_list((1, 2, 3))

[1, 2, 3]

In [16]:
sorted(as_list({3, 2, 1}))

[1, 2, 3]

In [17]:
# Generator
gen = (x * 2 for x in range(3))
as_list(gen)

[0, 2, 4]

Strings and bytes are treated as atomic and wrapped (['abc'], not ['a', 'b', 'c']).

In [18]:
as_list("abc")

['abc']

In [19]:
as_list(b"xyz")

[b'xyz']

For NumPy arrays, returns a list of scalar elements (if array is 1D).

In [20]:
import numpy as np

as_list(np.array([1, 2, 3]))

[1, 2, 3]

For PyTorch tensors, returns a list of subtensors along the first dimension ...

In [21]:
import torch

t = torch.tensor([[1, 2], [3, 4]])
as_list(t)

[tensor([1, 2]), tensor([3, 4])]

... to wrap PyTorch tensors in a list use `[t]`.

In [22]:
as_list([t])

[tensor([[1, 2],
         [3, 4]])]

## 4. Inspecting Function Argument Names with `get_kwarg_names()`

The `get_kwarg_names(function)` function returns a list of the argument names that can be provided as keyword arguments (including keyword-only arguments).

In [23]:
from deeptrack.utils import get_kwarg_names

In [24]:
# Normal function
def g(x, y):
    pass


get_kwarg_names(g)

['x', 'y']

In [25]:
# Function with optional arguments
def f(a, b=1, c=2):
    pass


get_kwarg_names(f)

['a', 'b', 'c']

In [26]:
# Function with *args and **kwargs
def k(*args, alpha=0.1, beta=0.2, **kwargs):
    pass


get_kwarg_names(k)

['alpha', 'beta']

In [27]:
# Built-in function
get_kwarg_names(len)

['obj']

In [28]:
# Lambda function
l = lambda x, y=5: x + y

get_kwarg_names(l)

['x', 'y']

In [29]:
# Method
class MyClass:
    def method(self, a, b=2):
        pass


get_kwarg_names(MyClass.method)

['self', 'a', 'b']

## 5. Determining If a Function Argument Has a Default Value with `kwarg_has_default()`

The `kwarg_has_default(function, argument)`function returns `True` if the specified argument of the function has a default value, `False` otherwise.

In [30]:
from deeptrack.utils import kwarg_has_default

It works with standard functions ...

In [31]:
def f(a, b=2, c=3):
    pass


kwarg_has_default(f, "a")

False

In [32]:
kwarg_has_default(f, "b")

True

In [33]:
kwarg_has_default(f, "c")

True

In [34]:
# Missing argument
kwarg_has_default(f, "not_present")

False

... and with methods.

In [35]:
class MyClass:
    def method(self, x, y=42):
        pass


kwarg_has_default(MyClass.method, "self")

False

In [36]:
kwarg_has_default(MyClass.method, "x")

False

In [37]:
kwarg_has_default(MyClass.method, "y")

True

## 6. Calling Functions Safely with Dictionaries of Arguments with `safe_call()`

The `safe_call(function, positional_args=None, **kwargs)` function calls the target function, only passing keyword arguments that are valid for that function. This is ideal for writing meta-programs, adapters, or any code that receives user arguments dynamically.

In [38]:
from deeptrack.utils import safe_call

In [39]:
def f(a, b=2, c=3):
    return a + b + c


# Pass a mix of positional and keyword arguments, extra args ignored
safe_call(f, positional_args=[1], b=4, x=100)

8

In [40]:
# All keyword arguments
safe_call(f, a=1, b=2, c=3)

6

In [41]:
# Extra keyword arguments (ignored)
safe_call(f, a=2, extra=42)

7

In [42]:
# Missing required argument (raises TypeError)
try:
    safe_call(f, b=2, c=3)
except TypeError as e:
    print("Caught error:", e)

Caught error: f() missing 1 required positional argument: 'a'


In [43]:
# Function with *args and **kwargs (only valid kwargs are passed)
def g(a, *args, b=5, **kwargs):
    return a, args, b, kwargs


safe_call(g, positional_args=[1, 10], b=7, x=3, y=2)

(1, (10,), 7, {})

In [44]:
# Function with only *args
def h(*args):
    return args


safe_call(h, positional_args=[1, 2, 3])

(1, 2, 3)